# Сборка аналитического датасета и EDA: ФП → Дефолт

**Контрольная точка:** КТ-3 (Результаты разведочного анализа)  
**Этап:** Шаг 3. Разведочный анализ и визуализация

---

### Цель ноутбука

Собрать единый аналитический датасет из трёх источников и провести разведочный анализ (EDA), чтобы ответить на вопрос: **какие факторы проблемности (ФП) связаны с дефолтом компании?**

### Единица наблюдения

**Компания (ИНН)**. Каждая строка итогового датасета — одна уникальная компания.

### Источники данных

| Источник | Роль | Таблица в Озере | Локальный файл | Строк | Колонок | Период |
|----------|------|----------------|----------------|------:|--------:|--------|
| **CRM** | **Основной** (ФП + вселенная) | `sandbox_ai.tmp_ecp_crm_fp_svy` | `data_crm.csv` | 1 504 263 | 41 | 2022-04 — 2025-12 |
| **H2.0** | Вспомогательный (сегмент) | `sandbox_ai.tmp_h20_fp_svy` | `data_h20.xlsx` | 172 981 | 11 | 2023-02 — 2025-12 |
| **Дефолты** | **Основной** (целевая переменная) | `sandbox_ai.tmp_test_default_svy` | `data_defaults.pkl` | 889 | 11 | 2020-01 — 2025-12 |

### Ключевые решения по архитектуре

1. **ФП берём из CRM** — колонка `NUMBER_FP_SFP` (тип ФП, 100% заполненность). Дата выявления — `IDENTIFICATION_DATE`.
2. **Вселенную компаний** берём из CRM (`X_INN`).
3. **H2O — вспомогательный источник**: используется для подтягивания сегмента бизнеса (`segment`).
4. **Временной фильтр**: ФП считаются только если выявлены **до** дефолта (или до отсечки для недефолтных) — исключает data leakage.
5. **Все компании** в выгрузке дефолтов получают `default_flag = 1`, тяжесть классифицируется отдельно.

### Структура итогового датасета

| Колонка | Тип | Описание |
|---------|-----|----------|
| `inn` | str | ИНН компании (ключ) |
| `default_flag` | int8 | 1 = дефолт, 0 = нет |
| `default_date` | datetime | Дата первого дефолта (NaT для недефолтных) |
| `severity` | str | Тяжесть: тяжёлый / активный / надзор / вышел из дефолта / нет дефолта |
| `segment` | str | Сегмент бизнеса клиента по обороту (из H2O) |
| `fp_*` | int8 | N бинарных колонок по числу уникальных NUMBER_FP_SFP: 1 = ФП был, 0 = нет |

### Разделы EDA (deliverables КТ-3)

| № | Раздел | Что выдаёт |
|---|--------|------------|
| 7 | Гистограммы ФП | Распределение числа ФП у дефолтных vs недефолтных |
| 8 | Динамика дефолтов | По месяцам и кварталам |
| 8.1 | Сегменты | Default rate по сегментам бизнеса |
| 9 | Тепловая карта | Корреляции между ТОП-30 ФП |
| 10 | Кросс-таблицы | Частота, доля дефолтов, Lift по каждому ФП |
| 10.1 | Комбинации | ТОП-5 пар и троек ФП по Lift |
| 11 | Аномалии | ФП с 100%/0% дефолтностью, выбросы, дефолты без ФП |
| 12 | Гипотезы | Направления для углублённого анализа (шаги 4–6) |

## 0. Импорты и настройки

In [ ]:
import pandas as pd
import numpy as np
import warnings

# Подавляем FutureWarning от pandas и DeprecationWarning от openpyxl,
# чтобы вывод ноутбука оставался чистым для заказчика
warnings.filterwarnings("ignore")

# Расширяем отображение таблиц в Jupyter, чтобы не терять колонки/строки
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

## 1. Загрузка данных

In [ ]:
# --- CRM: основной источник ФП ---
# Читаем как строки (dtype=str), чтобы избежать автоприведения ИНН/дат.
# Ключевые колонки: NUMBER_FP_SFP (тип ФП, 100% заполнен),
#                   IDENTIFICATION_DATE (дата выявления), X_INN (ИНН компании)
df_crm = pd.read_csv("../sources/data_crm.csv", sep=";", encoding="utf-8-sig", dtype=str, low_memory=False)
df_crm.columns = df_crm.columns.str.strip()
print(f"CRM:      {df_crm.shape[0]:>10,} строк × {df_crm.shape[1]} колонок")

# --- H2O: вспомогательный источник (для сегмента бизнеса) ---
df_h2o = pd.read_excel("../sources/data_h20.xlsx", dtype=str)
df_h2o.columns = df_h2o.columns.str.strip()
print(f"H2O:      {df_h2o.shape[0]:>10,} строк × {df_h2o.shape[1]} колонок")

# --- Дефолты: история дефолтов компаний ---
df_def = pd.read_pickle("../sources/data_defaults.pkl")
df_def = df_def.astype(str).replace("nan", np.nan)
df_def.columns = df_def.columns.str.strip()
print(f"Дефолты:  {df_def.shape[0]:>10,} строк × {df_def.shape[1]} колонок")

## 2. Подготовка таблицы дефолтов

Все компании в выгрузке defaults подверглись дефолту. Тяжесть:
- **writeoff=1** или **unlimited_default=1** → тяжёлый дефолт (списание / бессрочный)
- **finish_date заполнена** → компания вышла из дефолта (излечилась)
- **cure_date заполнена, finish_date пуста** → компания на стадии надзора (6 мес)
- Остальные → активный дефолт

`default_flag = 1` для **всех** компаний из этой выгрузки.

In [ ]:
# Приводим колонки дат к datetime.
# dayfirst=True — формат ДД.ММ.ГГГГ; errors="coerce" — битые значения → NaT
for col in ["start_date", "cure_date", "finish_date"]:
    if col in df_def.columns:
        df_def[col] = pd.to_datetime(df_def[col], dayfirst=True, errors="coerce")

# Приводим бинарные флаги к числовому типу (0/1)
for col in ["writeoff", "unlimited_default"]:
    if col in df_def.columns:
        df_def[col] = pd.to_numeric(df_def[col], errors="coerce").fillna(0).astype(int)


def classify_severity(row):
    """Классифицирует тяжесть дефолта по приоритету (от худшего к лучшему):
    
    1. «тяжёлый»          — списание (writeoff=1) или бессрочный дефолт (unlimited_default=1)
    2. «активный»         — компания в процессе дефолта (нет cure_date, нет finish_date)
    3. «надзор»           — есть cure_date (излечение), но finish_date ещё не наступила
                            (6 месяцев наблюдения после излечения)
    4. «вышел из дефолта» — finish_date заполнена (дефолт закрыт)
    """
    if row.get("writeoff") == 1 or row.get("unlimited_default") == 1:
        return "тяжёлый"
    if pd.notna(row.get("finish_date")):
        return "вышел из дефолта"
    if pd.notna(row.get("cure_date")):
        return "надзор"
    return "активный"


df_def["severity"] = df_def.apply(classify_severity, axis=1)

print(f"Всего строк дефолтов: {len(df_def):,}")
print(f"Уникальных ИНН в дефолтах: {df_def['inn'].nunique():,}")
print(f"Период: {df_def['start_date'].min()} — {df_def['start_date'].max()}")
print(f"\nРаспределение по тяжести:")
print(df_def["severity"].value_counts().to_string())

In [ ]:
# Агрегация до уровня компании: одна строка на ИНН.
# У одной компании может быть несколько записей о дефолте (повторные эпизоды).
# Логика: берём наихудшую тяжесть и самую раннюю дату начала дефолта.

# Приоритет тяжести: 0 = худший (тяжёлый), 3 = лучший (вышел)
severity_priority = {"тяжёлый": 0, "активный": 1, "надзор": 2, "вышел из дефолта": 3}
df_def["_sev_rank"] = df_def["severity"].map(severity_priority)

defaults = (
    df_def
    .dropna(subset=["inn", "start_date"])       # убираем строки без ИНН или даты
    .sort_values("_sev_rank")                    # сортируем: тяжёлые сверху
    .groupby("inn", as_index=False)
    .agg(
        default_date=("start_date", "min"),      # самая ранняя дата дефолта
        severity=("severity", "first"),           # наихудшая тяжесть (first после sort)
        writeoff=("writeoff", "max"),             # 1, если хотя бы один эпизод со списанием
        unlimited_default=("unlimited_default", "max"),  # 1, если есть бессрочный
    )
)
defaults["default_flag"] = 1  # все компании из этой таблицы — дефолтные

print(f"Уникальных компаний-дефолтников: {len(defaults):,}")
print(f"Период дефолтов: {defaults['default_date'].min()} — {defaults['default_date'].max()}")
print(f"\nТяжесть (на уровне компании):")
print(defaults["severity"].value_counts().to_string())

## 3. Подготовка таблицы ФП из CRM

`NUMBER_FP_SFP` — тип ФП (100% заполненность). `IDENTIFICATION_DATE` — дата выявления ФП.

CRM — мастер-система, содержит все ФП. Фильтруем по периоду 2023-02-01 — 2025-12-30 для согласованности с данными H2O.

In [ ]:
# Конвертируем дату выявления ФП в datetime
df_crm["IDENTIFICATION_DATE"] = pd.to_datetime(
    df_crm["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce"
)

# Фильтруем по периоду 2023-02-01 — 2025-12-30 (согласованность с H2O)
date_from = pd.Timestamp("2023-02-01")
date_to   = pd.Timestamp("2025-12-30")
df_crm = df_crm[
    (df_crm["IDENTIFICATION_DATE"] >= date_from) &
    (df_crm["IDENTIFICATION_DATE"] <= date_to)
].copy()

# Формируем таблицу ФП из CRM:
#   X_INN               → inn       (ИНН компании)
#   NUMBER_FP_SFP       → fp_type   (тип ФП, 100% заполнен)
#   IDENTIFICATION_DATE → fp_date   (дата выявления)
fp = df_crm[["X_INN", "NUMBER_FP_SFP", "IDENTIFICATION_DATE"]].copy()
fp = fp.rename(columns={
    "X_INN": "inn",
    "NUMBER_FP_SFP": "fp_type",
    "IDENTIFICATION_DATE": "fp_date",
})

fp = fp.dropna(subset=["inn", "fp_type"])

print(f"Строк ФП (CRM, после фильтра дат): {len(fp):,}")
print(f"Уникальных типов ФП (NUMBER_FP_SFP): {fp['fp_type'].nunique()}")
print(f"Уникальных ИНН в CRM: {fp['inn'].nunique():,}")
print(f"Период ФП: {fp['fp_date'].min()} — {fp['fp_date'].max()}")

## 4. Вселенная компаний

Вселенную берём из **CRM** (мастер-система). Присоединяем флаг и тяжесть дефолта.

In [ ]:
# Уникальные ИНН из CRM (мастер-система)
universe = fp[["inn"]].drop_duplicates().copy()

# LEFT JOIN с дефолтами: компании из defaults получают default_flag=1,
# остальные — default_flag=0, severity="нет дефолта"
universe = universe.merge(
    defaults[["inn", "default_flag", "default_date", "severity"]],
    on="inn", how="left"
)
universe["default_flag"] = universe["default_flag"].fillna(0).astype(int)
universe["severity"] = universe["severity"].fillna("нет дефолта")

print(f"Всего уникальных компаний: {len(universe):,}")
print(f"Из них дефолтных:          {universe['default_flag'].sum():,}")
print(f"Default rate:              {universe['default_flag'].mean():.2%}")
print(f"\nРаспределение по severity:")
print(universe["severity"].value_counts().to_string())

## 5. Временной фильтр ФП

**Критично для корректности:**
- Для дефолтных: оставляем ФП, выявленные **до** даты дефолта
- Для недефолтных: оставляем ФП, выявленные **до** даты отсечки

In [ ]:
# Дата отсечки для недефолтных компаний: конец периода наблюдения.
# Для них учитываем все ФП, выявленные до этой даты.
CUTOFF = pd.Timestamp("2025-12-31")

# Присоединяем к каждому ФП информацию о дефолте компании
fp_with_def = fp.merge(
    universe[["inn", "default_flag", "default_date"]], on="inn", how="inner"
)

# Референсная дата: дефолтные → дата дефолта, недефолтные → CUTOFF
fp_with_def["ref_date"] = fp_with_def["default_date"].fillna(CUTOFF)

# ГЛАВНЫЙ ФИЛЬТР: оставляем только ФП, выявленные СТРОГО ДО референсной даты.
# Это исключает data leakage — ФП, появившиеся после дефолта, не могли его «предсказать».
before = len(fp_with_def)
fp_filtered = fp_with_def[fp_with_def["fp_date"] < fp_with_def["ref_date"]].copy()
after = len(fp_filtered)

print(f"ФП до временного фильтра:  {before:,}")
print(f"ФП после временного фильтра: {after:,}")
print(f"Отсечено (ФП после дефолта): {before - after:,}")

## 6. Pivot — широкий формат

In [ ]:
# Вспомогательная колонка для pivot: наличие ФП = 1
fp_filtered["value"] = 1

# Pivot: длинная таблица (1 строка = 1 ФП) → широкая (1 строка = 1 компания).
# Колонки = типы ФП (NUMBER_FP_SFP), значения = 0/1.
# aggfunc="max": если один тип ФП встретился несколько раз — всё равно 1.
fp_wide = (
    fp_filtered
    .pivot_table(index="inn", columns="fp_type", values="value",
                 aggfunc="max", fill_value=0)
    .reset_index()
)
fp_wide.columns.name = None

# Подтягиваем сегмент бизнеса из H2O (вспомогательный источник).
# У одной компании может быть несколько записей → берём самый частый сегмент.
segment_per_inn = (
    df_h2o[["inn", "segment"]]
    .dropna(subset=["inn", "segment"])
    .groupby("inn")["segment"]
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
)

# Собираем итоговый датасет: вселенная + wide ФП + сегмент
meta_cols = ["inn", "default_flag", "default_date", "severity"]
dataset = universe[meta_cols].merge(fp_wide, on="inn", how="left")  # LEFT JOIN: компании без ФП тоже попадают
dataset = dataset.merge(segment_per_inn, on="inn", how="left")
dataset["segment"] = dataset["segment"].fillna("Неизвестно")

# Определяем колонки ФП (всё, что не мета-данные)
meta_cols = ["inn", "default_flag", "default_date", "severity", "segment"]
fp_cols = [c for c in dataset.columns if c not in meta_cols]

# Компании без единого ФП → NaN в fp-колонках → заполняем нулями.
# int8 вместо int64 экономит память в ~8 раз (1 байт vs 8 байт на ячейку).
dataset[fp_cols] = dataset[fp_cols].fillna(0).astype("int8")
dataset["default_flag"] = dataset["default_flag"].astype("int8")

print(f"Итоговый датасет: {dataset.shape[0]:,} строк × {dataset.shape[1]} колонок")
print(f"  — компаний:     {dataset.shape[0]:,}")
print(f"  — дефолтных:    {dataset['default_flag'].sum():,}")
print(f"  — ФП-колонок:   {len(fp_cols)}")
print(f"  — сегментов:    {dataset['segment'].nunique()}")
print(f"  — в памяти:     {dataset.memory_usage(deep=True).sum() / 1e6:.1f} МБ")
print(f"\nДефолтные по severity:")
print(dataset[dataset["default_flag"]==1]["severity"].value_counts().to_string())

## 7. Гистограммы распределения ФП: дефолтные vs недефолтные

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Считаем общее число ФП на каждую компанию (сумма по 273 бинарным колонкам)
fp_per_company = dataset[fp_cols].sum(axis=1)
dataset["fp_count"] = fp_per_company

# Разделяем на дефолтных и недефолтных для сравнения распределений
def_counts = fp_per_company[dataset["default_flag"] == 1]
nodef_counts = fp_per_company[dataset["default_flag"] == 0]

# Таблица со сводной статистикой
print("Число ФП на компанию:")
print(f"  {'':20s} {'Дефолтные':>12s} {'Недефолтные':>12s}")
print(f"  {'Среднее':20s} {def_counts.mean():>12.1f} {nodef_counts.mean():>12.1f}")
print(f"  {'Медиана':20s} {def_counts.median():>12.0f} {nodef_counts.median():>12.0f}")
print(f"  {'Макс':20s} {def_counts.max():>12.0f} {nodef_counts.max():>12.0f}")
print(f"  {'Без ФП':20s} {(def_counts == 0).sum():>12,} {(nodef_counts == 0).sum():>12,}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Обрезаем по 95-му перцентилю, чтобы выбросы не сжимали основную часть гистограммы
max_fp = int(fp_per_company.quantile(0.95)) + 1
bins = range(0, max_fp + 2)

# Левый график: абсолютные числа (показывает масштаб дисбаланса классов)
axes[0].hist(nodef_counts.clip(upper=max_fp), bins=bins, alpha=0.7, label="Недефолтные", color="steelblue", edgecolor="white")
axes[0].hist(def_counts.clip(upper=max_fp), bins=bins, alpha=0.7, label="Дефолтные", color="tomato", edgecolor="white")
axes[0].set_xlabel("Количество ФП")
axes[0].set_ylabel("Число компаний")
axes[0].set_title("Распределение количества ФП")
axes[0].legend()

# Правый график: плотность (нормализованные, чтобы сравнивать форму распределений)
axes[1].hist(nodef_counts.clip(upper=max_fp), bins=bins, alpha=0.7, density=True, label="Недефолтные", color="steelblue", edgecolor="white")
axes[1].hist(def_counts.clip(upper=max_fp), bins=bins, alpha=0.7, density=True, label="Дефолтные", color="tomato", edgecolor="white")
axes[1].set_xlabel("Количество ФП")
axes[1].set_ylabel("Доля")
axes[1].set_title("Нормализованное распределение (плотность)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Динамика дефолтов по месяцам/кварталам

In [ ]:
# Берём только дефолтные компании и создаём агрегаты по периодам
defaults_ts = dataset[dataset["default_flag"] == 1].copy()
defaults_ts["default_month"] = defaults_ts["default_date"].dt.to_period("M")
defaults_ts["default_quarter"] = defaults_ts["default_date"].dt.to_period("Q")

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Верхний график: помесячная динамика (детальная, видны всплески)
monthly = defaults_ts.groupby("default_month").size()
monthly.index = monthly.index.astype(str)
axes[0].bar(range(len(monthly)), monthly.values, color="tomato", edgecolor="white")
axes[0].set_xticks(range(len(monthly)))
axes[0].set_xticklabels(monthly.index, rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Количество дефолтов")
axes[0].set_title("Динамика дефолтов по месяцам")

# Нижний график: поквартальная динамика (сглаженная, видны тренды)
quarterly = defaults_ts.groupby("default_quarter").size()
quarterly.index = quarterly.index.astype(str)
axes[1].bar(range(len(quarterly)), quarterly.values, color="coral", edgecolor="white")
axes[1].set_xticks(range(len(quarterly)))
axes[1].set_xticklabels(quarterly.index, rotation=45, ha="right")
axes[1].set_ylabel("Количество дефолтов")
axes[1].set_title("Динамика дефолтов по кварталам")

plt.tight_layout()
plt.show()

print("\nДефолты по кварталам:")
print(quarterly.to_string())

## 8.1. Дефолтность по сегментам бизнеса

In [ ]:
# Агрегируем статистику дефолтов по каждому сегменту бизнеса:
# — количество компаний, количество дефолтов, default rate, доля от всех дефолтов
seg_stats = (
    dataset.groupby("segment")
    .agg(
        компаний=("default_flag", "size"),
        дефолтов=("default_flag", "sum"),
    )
    .assign(
        default_rate=lambda x: (x["дефолтов"] / x["компаний"] * 100).round(2),
        доля_от_всех_дефолтов=lambda x: (x["дефолтов"] / x["дефолтов"].sum() * 100).round(1),
    )
    .sort_values("default_rate", ascending=False)
    .reset_index()
)

print("Дефолтность по сегментам бизнеса:")
display(
    seg_stats
    .style.hide(axis="index")
    .format({"default_rate": "{:.2f}%", "доля_от_всех_дефолтов": "{:.1f}%"})
    .bar(subset=["default_rate"], color="salmon")
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seg_plot = seg_stats.sort_values("default_rate", ascending=True)

# Левый график: default rate — показывает «опасность» сегмента
axes[0].barh(seg_plot["segment"], seg_plot["default_rate"], color="tomato", edgecolor="white")
axes[0].set_xlabel("Default rate, %")
axes[0].set_title("Доля дефолтов по сегментам")

# Правый график: размер сегмента — контекст для интерпретации
# (высокий default rate при 3 компаниях ≠ высокий default rate при 3000)
axes[1].barh(seg_plot["segment"], seg_plot["компаний"], color="steelblue", edgecolor="white")
axes[1].set_xlabel("Количество компаний")
axes[1].set_title("Размер сегмента (число компаний)")

plt.tight_layout()
plt.show()

# Кросс-таблица: тяжесть дефолта × сегмент
seg_severity = (
    dataset[dataset["default_flag"] == 1]
    .groupby(["segment", "severity"]).size()
    .unstack(fill_value=0)
)
print("\nТяжесть дефолта по сегментам:")
display(seg_severity)

## 9. Тепловая карта корреляций между ФП

Корреляция Phi (эквивалент Пирсона для бинарных признаков) между наиболее частыми ФП. Показывает, какие ФП часто встречаются вместе.

In [ ]:
import seaborn as sns

# Строим тепловую карту только для ТОП-30 самых частых ФП,
# чтобы матрица 273×273 не была нечитаемой
TOP_N_HEATMAP = 30
fp_freq = dataset[fp_cols].sum().sort_values(ascending=False)
top_fp = fp_freq.head(TOP_N_HEATMAP).index.tolist()

# Корреляция Пирсона между бинарными признаками = Phi-коэффициент.
# Показывает, какие ФП часто встречаются вместе (положительная корреляция)
# или взаимоисключают друг друга (отрицательная).
corr = dataset[top_fp].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    corr, annot=False, cmap="RdBu_r", center=0,
    vmin=-0.5, vmax=0.5, linewidths=0.3,
    xticklabels=True, yticklabels=True, ax=ax,
)
ax.set_title(f"Корреляции между ТОП-{TOP_N_HEATMAP} наиболее частыми ФП")
plt.xticks(fontsize=7, rotation=90)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()

# Извлекаем верхний треугольник матрицы (без диагонали) в плоский формат,
# чтобы найти пары с сильной корреляцией
high_corr = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
high_corr.columns = ["ФП_1", "ФП_2", "Корреляция"]
high_corr["abs_corr"] = high_corr["Корреляция"].abs()

# Порог |r| > 0.3: сильная для бинарных признаков (Phi обычно ниже Пирсона)
print(f"\nПары ФП с наибольшей корреляцией (|r| > 0.3):")
strong = high_corr[high_corr["abs_corr"] > 0.3].sort_values("abs_corr", ascending=False)
if len(strong) > 0:
    display(strong.head(20).style.hide(axis="index").format({"Корреляция": "{:.3f}", "abs_corr": "{:.3f}"}))
else:
    print("  Пар с |r| > 0.3 не найдено")

## 10. Кросс-таблицы «ФП × Дефолт» и первичные находки по каждому типу ФП

Для каждого ФП:
- **Частота встречаемости** — у скольких компаний есть этот ФП
- **Доля дефолтов** — какой % носителей ФП оказался в дефолте
- **Lift** — во сколько раз доля дефолтов с ФП выше базовой (предварительная оценка «опасности»)

In [ ]:
# Базовый default rate — «фон», относительно которого оцениваем каждый ФП
base_rate = dataset["default_flag"].mean()
total_companies = len(dataset)
total_defaults = dataset["default_flag"].sum()

# Для каждого из 273 ФП считаем:
#   n_with         — сколько компаний имеют этот ФП
#   n_defaults_with — сколько из них дефолтных
#   rate_with      — доля дефолтов среди носителей ФП
#   lift           — во сколько раз rate_with выше базового default rate
#                    (lift > 1 = ФП «опасный», lift < 1 = ФП «безопасный»)
results = []
for col in fp_cols:
    n_with = (dataset[col] == 1).sum()
    n_defaults_with = dataset.loc[dataset[col] == 1, "default_flag"].sum()
    rate_with = n_defaults_with / n_with if n_with > 0 else 0
    lift = rate_with / base_rate if base_rate > 0 else 0

    results.append({
        "Тип ФП": col,
        "Компаний с ФП": n_with,
        "% от всех компаний": round(n_with / total_companies * 100, 2),
        "Дефолтов среди них": n_defaults_with,
        "Доля дефолтов": round(rate_with * 100, 2),
        "Lift": round(lift, 2),
    })

df_cross = pd.DataFrame(results).sort_values("Lift", ascending=False)

print(f"Базовый default rate: {base_rate:.2%}")
print(f"Всего компаний: {total_companies:,}, дефолтов: {total_defaults:,}\n")

# Показываем ТОП-30, фильтруя ФП с < 3 дефолтами (иначе lift ненадёжен)
print("="*70)
print("ТОП-30 ФП по Lift (предварительная оценка «опасности»):")
print("="*70)
display(
    df_cross[df_cross["Дефолтов среди них"] >= 3]
    .head(30)
    .style.hide(axis="index")
    .format({"Доля дефолтов": "{:.2f}%", "% от всех компаний": "{:.2f}%"})
    .bar(subset=["Lift"], color="salmon")
)

In [ ]:
print("="*70)
print("Полная кросс-таблица «ФП × Дефолт» (все 273 ФП):")
print("="*70)
display(
    df_cross
    .style.hide(axis="index")
    .format({"Доля дефолтов": "{:.2f}%", "% от всех компаний": "{:.2f}%"})
)

## 10.1. Топ-5 комбинаций из 2–3 ФП, ведущих к дефолту

Перебираем пары и тройки **только** среди ФП с достаточной частотой (≥20 компаний). Для каждой комбинации считаем lift = (default rate комбинации) / (базовый default rate).

In [ ]:
from itertools import combinations

# Пороги для отсечки шумных комбинаций:
# — минимум 10 компаний с данной комбинацией ФП
# — минимум 3 дефолта среди них (иначе lift = случайность)
MIN_COMPANIES_COMBO = 10
MIN_DEFAULTS_COMBO = 3

# Отбираем только достаточно частые ФП (≥20 компаний).
# Для редких ФП комбинации почти всегда будут пустыми.
frequent_fp = [c for c in fp_cols if (dataset[c] == 1).sum() >= 20]
print(f"ФП с ≥20 компаниями (для поиска комбинаций): {len(frequent_fp)}")

# Переводим в numpy-массивы для скорости перебора
base_rate = dataset["default_flag"].mean()
y = dataset["default_flag"].values
X = dataset[frequent_fp].values
fp_names = np.array(frequent_fp)

# --- Пары (C(n,2) комбинаций) ---
# Для каждой пары: считаем компании, где оба ФП = 1, и долю дефолтов среди них
pair_results = []
for i, j in combinations(range(len(frequent_fp)), 2):
    mask = (X[:, i] == 1) & (X[:, j] == 1)
    n = mask.sum()
    if n < MIN_COMPANIES_COMBO:
        continue
    n_def = y[mask].sum()
    if n_def < MIN_DEFAULTS_COMBO:
        continue
    rate = n_def / n
    lift = rate / base_rate
    pair_results.append({
        "Комбинация": f"{fp_names[i]}  +  {fp_names[j]}",
        "Размер": 2,
        "Компаний": n,
        "Дефолтов": n_def,
        "Default rate": round(rate * 100, 2),
        "Lift": round(lift, 2),
    })

df_pairs = pd.DataFrame(pair_results).sort_values("Lift", ascending=False)
print(f"Найдено пар с ≥{MIN_COMPANIES_COMBO} компаний и ≥{MIN_DEFAULTS_COMBO} дефолтов: {len(df_pairs)}")

print(f"\n{'='*70}")
print("ТОП-5 пар ФП по Lift:")
print("="*70)
if len(df_pairs) > 0:
    display(
        df_pairs.head(5)
        .style.hide(axis="index")
        .format({"Default rate": "{:.2f}%"})
        .bar(subset=["Lift"], color="salmon")
    )
else:
    print("  Пар, удовлетворяющих порогам, не найдено.")

In [ ]:
# --- Тройки (C(n,3) комбинаций) ---
# Полный перебор троек из 273 ФП = ~3.3M — слишком долго.
# Ограничиваем: берём только ТОП-30 ФП по индивидуальному lift.
# C(30,3) = 4060 — вполне быстро.
top_individual = (
    df_cross[df_cross["Компаний с ФП"] >= 20]
    .sort_values("Lift", ascending=False)
    .head(30)["Тип ФП"]
    .tolist()
)
top_idx = [i for i, name in enumerate(frequent_fp) if name in top_individual]

triple_results = []
for i, j, k in combinations(top_idx, 3):
    mask = (X[:, i] == 1) & (X[:, j] == 1) & (X[:, k] == 1)
    n = mask.sum()
    if n < MIN_COMPANIES_COMBO:
        continue
    n_def = y[mask].sum()
    if n_def < MIN_DEFAULTS_COMBO:
        continue
    rate = n_def / n
    lift = rate / base_rate
    triple_results.append({
        "Комбинация": f"{fp_names[i]}  +  {fp_names[j]}  +  {fp_names[k]}",
        "Размер": 3,
        "Компаний": n,
        "Дефолтов": n_def,
        "Default rate": round(rate * 100, 2),
        "Lift": round(lift, 2),
    })

df_triples = pd.DataFrame(triple_results).sort_values("Lift", ascending=False) if triple_results else pd.DataFrame()
print(f"Найдено троек с ≥{MIN_COMPANIES_COMBO} компаний и ≥{MIN_DEFAULTS_COMBO} дефолтов: {len(df_triples)}")

print(f"\n{'='*70}")
print("ТОП-5 троек ФП по Lift:")
print("="*70)
if len(df_triples) > 0:
    display(
        df_triples.head(5)
        .style.hide(axis="index")
        .format({"Default rate": "{:.2f}%"})
        .bar(subset=["Lift"], color="salmon")
    )
else:
    print("  Троек, удовлетворяющих порогам, не найдено.")

# --- Итоговая сводка: объединяем пары и тройки, показываем ТОП-5 ---
df_all_combos = pd.concat([df_pairs, df_triples], ignore_index=True).sort_values("Lift", ascending=False)
print(f"\n{'='*70}")
print("ТОП-5 комбинаций (пары + тройки) по Lift:")
print("="*70)
if len(df_all_combos) > 0:
    display(
        df_all_combos.head(5)
        .style.hide(axis="index")
        .format({"Default rate": "{:.2f}%"})
        .bar(subset=["Lift"], color="salmon")
    )

## 11. Аномалии и интересные паттерны

In [ ]:
print("="*70)
print("АНОМАЛИИ И ПАТТЕРНЫ")
print("="*70)

# 1. ФП, при которых ВСЕ носители оказались дефолтными (100% default rate).
#    Порог ≥2 компании — при 1 компании 100% ничего не значит.
fp_100 = df_cross[(df_cross["Доля дефолтов"] == 100) & (df_cross["Компаний с ФП"] >= 2)]
print(f"\n1. ФП с 100% дефолтностью (≥2 компании): {len(fp_100)}")
if len(fp_100) > 0:
    display(fp_100.style.hide(axis="index"))

# 2. ФП, при которых ни один носитель не дефолтнул (0% при ≥20 компаниях).
#    Потенциально «безопасные» ФП — или просто не связанные с дефолтом.
fp_0 = df_cross[(df_cross["Доля дефолтов"] == 0) & (df_cross["Компаний с ФП"] >= 20)]
print(f"\n2. ФП с 0% дефолтностью (≥20 компаний): {len(fp_0)}")
if len(fp_0) > 0:
    display(fp_0.style.hide(axis="index"))

# 3. Редкие ФП (< 5 компаний) — по ним невозможно делать выводы,
#    стоит рассмотреть группировку в категории на следующих шагах.
rare_fp = df_cross[df_cross["Компаний с ФП"] < 5]
print(f"\n3. Редкие ФП (< 5 компаний): {len(rare_fp)} из {len(df_cross)}")

# 4. Компании-выбросы по числу ФП (> 99-го перцентиля).
#    Интересно: дефолтные они или нет?
q99 = fp_per_company.quantile(0.99)
outliers = dataset[fp_per_company > q99]
print(f"\n4. Компании с аномально большим числом ФП (> {q99:.0f}, 99-й перцентиль): {len(outliers)}")
if len(outliers) > 0:
    outlier_info = outliers[["inn", "default_flag", "severity", "fp_count"]].sort_values("fp_count", ascending=False)
    display(outlier_info.head(10).style.hide(axis="index"))

# 5. Дефолтные компании, у которых вообще нет ФП в CRM за анализируемый период.
#    Возможные причины: ФП выявлены после дефолта (отсечены временным фильтром),
#    или не заводились вовсе.
def_no_fp = dataset[(dataset["default_flag"] == 1) & (dataset["fp_count"] == 0)]
print(f"\n5. Дефолтные компании без единого ФП: {len(def_no_fp)}")
if len(def_no_fp) > 0:
    print(f"   Это {len(def_no_fp) / dataset['default_flag'].sum():.1%} от всех дефолтных")

## 12. Гипотезы для углублённого анализа и рекомендации по приоритизации

In [ ]:
print("="*70)
print("ГИПОТЕЗЫ ДЛЯ УГЛУБЛЁННОГО АНАЛИЗА")
print("="*70)

n_high_lift = len(df_cross[(df_cross["Lift"] > 2) & (df_cross["Дефолтов среди них"] >= 5)])
n_corr_pairs = len(strong) if len(strong) > 0 else 0
n_def_no_fp = len(def_no_fp)
mean_fp_def = def_counts.mean()
mean_fp_nodef = nodef_counts.mean()

hypotheses = [
    f"Г1. Комбинации ФП: {n_corr_pairs} пар(ы) ФП с сильной корреляцией — "
    f"проверить, повышают ли они риск дефолта совместно (Apriori / lift комбинаций).",

    f"Г2. «Опасные» ФП: {n_high_lift} ФП с lift > 2 и ≥5 дефолтами — "
    f"проверить статистическую значимость (Fisher exact test, FDR-коррекция).",

    f"Г3. Количество ФП как предиктор: у дефолтных в среднем {mean_fp_def:.1f} ФП, "
    f"у недефолтных {mean_fp_nodef:.1f} — проверить, есть ли пороговое значение.",
]

if n_def_no_fp > 0:
    hypotheses.append(
        f"Г4. Дефолты без ФП: {n_def_no_fp} компаний дефолтнули без единого ФП в CRM — "
        f"возможно, ФП выявлены после дефолта или не заводились."
    )

for h in hypotheses:
    print(f"\n  {h}")

print(f"\n{'='*70}")
print("РЕКОМЕНДАЦИИ ПО ПРИОРИТИЗАЦИИ")
print("="*70)

recommendations = [
    "1. На следующем шаге (Гипотеза 1) — сфокусироваться на ФП с lift > 2 и ≥5 дефолтами, "
    "проверить их значимость и искать комбинации среди них.",

    "2. Редкие ФП (< 5 компаний) — не анализировать отдельно, "
    "рассмотреть группировку в категории.",

    "3. ФП с высокой взаимной корреляцией — учесть при моделировании "
    "(мультиколлинеарность), возможно объединить в группы.",

    "4. Проверить временную стабильность результатов: "
    "сохраняется ли рейтинг «опасности» ФП при разбиении 2022-2024 / 2025.",
]

for r in recommendations:
    print(f"\n  {r}")

## 13. Сохранение датасета и результатов EDA

In [ ]:
# Сохраняем два артефакта:
# 1. Полный датасет (pickle) — для использования на следующих шагах (Гипотезы 1-3)
# 2. Кросс-таблица (Excel) — для передачи заказчикам как часть отчёта КТ-3
dataset.to_pickle("../sources/dataset_fp_default.pkl")
df_cross.to_excel("../sources/eda_cross_table_fp_default.xlsx", index=False)

print("Сохранено:")
print("  ../sources/dataset_fp_default.pkl          — итоговый датасет (компания × ФП)")
print("  ../sources/eda_cross_table_fp_default.xlsx  — кросс-таблица ФП×Дефолт с частотами и lift")